In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import sqlite3
import pandas as pd

# ==================== ЗАДАНИЕ 1 ==================================

conn = sqlite3.connect(r"C:\Users\gmamy\Downloads\bank_analytics_training.db")

# DataFrame активности клиентов
query_client_activity = """
SELECT 
    SUBSTR(transaction_date, 1, 4) AS year,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount,
    AVG(amount) AS avg_amount
FROM transactions
GROUP BY year
ORDER BY year;
"""
df_client_activity = pd.read_sql(query_client_activity, conn)

df_client_activity

# DataFrame топ клиентов
query_top_clients = """
SELECT 
    customer_id,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount
FROM transactions
GROUP BY customer_id
ORDER BY total_amount DESC
LIMIT 20;
"""

df_top_clients = pd.read_sql(query_top_clients, conn)

df_top_clients

# дополнения для Dashboard
# DataFrame каналов
query_channels = """
SELECT 
    channel,
    COUNT(*) AS transaction_count
FROM transactions
GROUP BY channel
ORDER BY transaction_count DESC;
"""

df_channels = pd.read_sql(query_channels, conn)

# DataFrame валют
query_currency = """
SELECT 
    currency,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount
FROM transactions
GROUP BY currency
ORDER BY total_amount DESC;
"""

df_currency = pd.read_sql(query_currency, conn)

with pd.ExcelWriter("task1_transactions_report.xlsx") as writer:
    df_client_activity.to_excel(writer, sheet_name="client_activity", index=False)
    df_top_clients.to_excel(writer, sheet_name="top_clients", index=False)
    df_channels.to_excel(writer, sheet_name="channels", index=False)
    df_currency.to_excel(writer, sheet_name="currency", index=False)

df_client_activity.to_csv("client_activity.csv", index=False)
df_top_clients.to_csv("top_clients.csv", index=False)
df_channels.to_csv("channels.csv", index=False)
df_currency.to_csv("currency.csv", index=False)



In [26]:
import sqlite3
import pandas as pd

# ==================== ЗАДАНИЕ 2 ==================================

conn = sqlite3.connect(r"C:\Users\gmamy\Downloads\bank_analytics_training.db")

# таблица кредитного портфеля
query_loans = """
SELECT 
    loan_id,
    customer_id,
    branch_id,
    employee_id,
    product_name,
    loan_amount,
    interest_rate,
    status,
    issue_date
FROM loans;
"""
df_loans = pd.read_sql(query_loans, conn)
df_loans

# таблица просрочек
query_overdue = """
SELECT 
    l.loan_id,
    l.customer_id,
    l.branch_id,
    b.branch_name,
    l.employee_id,
    e.full_name AS employee_name,
    l.product_name,
    l.loan_amount,
    l.interest_rate,
    l.status,
    l.issue_date
FROM loans l
LEFT JOIN branches b
    ON l.branch_id = b.branch_id
LEFT JOIN employees e
    ON l.employee_id = e.employee_id
WHERE l.status = 'OVERDUE';
"""

df_overdue = pd.read_sql(query_overdue, conn)
df_overdue

# Рассчеты:
# - overdue rate
# - average loan amount
# - average interest rate

total_loans = len(df_loans)
total_overdue = len(df_overdue)

overdue_rate = total_overdue / total_loans * 100
average_loan_amount = df_loans["loan_amount"].mean()
average_interest_rate = df_loans["interest_rate"].mean()

print("Overdue rate:", overdue_rate)
print("Average loan amount:", average_loan_amount)
print("Average interest rate:", average_interest_rate)

query_branches = """
SELECT 
    b.branch_id,
    b.branch_name,
    COUNT(l.loan_id) AS loan_count,
    SUM(l.loan_amount) AS total_loan_amount
FROM branches b
LEFT JOIN loans l
    ON b.branch_id = l.branch_id
GROUP BY b.branch_id, b.branch_name;
"""
df_branches = pd.read_sql(query_branches, conn)


query_employees = """
SELECT 
    e.employee_id,
    e.full_name,
    COUNT(l.loan_id) AS loan_count,
    SUM(l.loan_amount) AS total_loan_amount
FROM employees e
LEFT JOIN loans l
    ON e.employee_id = l.employee_id
GROUP BY e.employee_id, e.full_name;
"""
df_employees = pd.read_sql(query_employees, conn)

# дополнения для Dashboard

df_kpi = pd.DataFrame({
    "total_loans": [total_loans],
    "total_overdue": [total_overdue],
    "overdue_rate": [overdue_rate],
    "average_loan_amount": [average_loan_amount]
})

query_loan_status = """
SELECT 
    status,
    COUNT(*) AS loan_count
FROM loans
GROUP BY status;
"""
df_loan_status = pd.read_sql(query_loan_status, conn)

query_loans_by_year = """
SELECT 
    SUBSTR(issue_date, 1, 4) AS year,
    COUNT(*) AS loan_count,
    SUM(loan_amount) AS total_loan_amount
FROM loans
GROUP BY year
ORDER BY year;
"""
df_loans_by_year = pd.read_sql(query_loans_by_year, conn)

with pd.ExcelWriter("task2_credit_report.xlsx") as writer:
    df_loans.to_excel(writer, sheet_name="loans", index=False)
    df_overdue.to_excel(writer, sheet_name="overdue", index=False)
    df_branches.to_excel(writer, sheet_name="branches", index=False)
    df_employees.to_excel(writer, sheet_name="employees", index=False)
    df_kpi.to_excel(writer, sheet_name="kpi", index=False)
    df_loan_status.to_excel(writer, sheet_name="loan_status", index=False)
    df_loans_by_year.to_excel(writer, sheet_name="loans_by_year", index=False)



Overdue rate: 25.577119509703778
Average loan amount: 250739.51660878447
Average interest rate: 16.102151174668027


In [27]:
import sqlite3
import pandas as pd

# ==================== ЗАДАНИЕ 3 ==================================

conn = sqlite3.connect(r"C:\Users\gmamy\Downloads\bank_analytics_training.db")

# Финальный DataFrame

df_customers = pd.read_sql("SELECT customer_id, full_name, city FROM customers", conn)
df_accounts = pd.read_sql("SELECT account_id, customer_id FROM accounts", conn)
df_cards = pd.read_sql("SELECT card_id, customer_id FROM cards", conn)
df_transactions = pd.read_sql("SELECT transaction_id, customer_id, amount, transaction_date, channel FROM transactions", conn)
df_loans = pd.read_sql("SELECT loan_id, customer_id, loan_amount, status FROM loans", conn)
df_applications = pd.read_sql("SELECT customer_id, decision_status, decision_date FROM applications", conn)

# Выполнение:
# - merge таблиц
# - groupby
# - agg
# - pivot table
accounts_agg = df_accounts.groupby("customer_id").agg(
    account_count=("account_id", "nunique")
).reset_index()

cards_agg = df_cards.groupby("customer_id").agg(
    card_count=("card_id", "nunique")
).reset_index()

transactions_agg = df_transactions.groupby("customer_id").agg(
    transaction_count=("transaction_id", "nunique"),
    transaction_amount=("amount", "sum"),
    last_transaction_date=("transaction_date", "max")
).reset_index()

loans_agg = df_loans.groupby("customer_id").agg(
    loan_count=("loan_id", "nunique"),
    loan_amount=("loan_amount", "sum"),
    overdue_count=("status", lambda x: (x == "OVERDUE").sum())
).reset_index()

df_applications_sorted = df_applications.sort_values("decision_date", ascending=False)

last_applications = df_applications_sorted.drop_duplicates("customer_id")

last_applications = last_applications[["customer_id", "decision_status"]]
last_applications = last_applications.rename(columns={"decision_status": "last_application_status"})

final_df = df_customers.merge(accounts_agg, on="customer_id", how="left")
final_df = final_df.merge(cards_agg, on="customer_id", how="left")
final_df = final_df.merge(transactions_agg, on="customer_id", how="left")
final_df = final_df.merge(loans_agg, on="customer_id", how="left")
final_df = final_df.merge(last_applications, on="customer_id", how="left")

final_df[["account_count", "card_count", "transaction_count", "transaction_amount", "loan_count", "loan_amount", "overdue_count"]] = final_df[[
    "account_count", "card_count", "transaction_count", "transaction_amount", "loan_count", "loan_amount", "overdue_count"
]].fillna(0)

# Создание:
# - отдельные DataFrame по сегментам клиентов
def segment_customer(row):
    if row["transaction_amount"] > 1000000:
        return "VIP"
    elif row["transaction_count"] > 50:
        return "Active"
    elif row["overdue_count"] > 0:
        return "Risk"
    else:
        return "Regular"

final_df["customer_segment"] = final_df.apply(segment_customer, axis=1)

final_df

segment_pivot = final_df.pivot_table(
    index="customer_segment",
    values="customer_id",
    aggfunc="count"
).reset_index()

segment_pivot = segment_pivot.rename(columns={"customer_id": "customer_count"})

segment_pivot

df_vip = final_df[final_df["customer_segment"] == "VIP"]
df_active = final_df[final_df["customer_segment"] == "Active"]
df_risk = final_df[final_df["customer_segment"] == "Risk"]
df_regular = final_df[final_df["customer_segment"] == "Regular"]

# дополнения для Dashboard

df_kpi = pd.DataFrame({
    "total_customers": [final_df["customer_id"].nunique()],
    "active_customers": [(final_df["transaction_count"] > 50).sum()],
    "total_transaction_volume": [final_df["transaction_amount"].sum()],
    "total_loans": [final_df["loan_count"].sum()],
    "overdue_loans": [final_df["overdue_count"].sum()],
    "vip_clients": [(final_df["customer_segment"] == "VIP").sum()]
})

city_pivot = final_df.groupby("city").agg(
    customer_count=("customer_id", "nunique")
).reset_index()

loan_status_pivot = df_loans.groupby("status").agg(
    loan_count=("loan_id", "nunique")
).reset_index()

channel_pivot = df_transactions.groupby("channel").agg(
    transaction_count=("transaction_id", "nunique"),
    total_amount=("amount", "sum")
).reset_index()

activity_by_year = df_transactions.copy()
activity_by_year["year"] = activity_by_year["transaction_date"].str[:4]

activity_by_year = activity_by_year.groupby("year").agg(
    transaction_count=("transaction_id", "nunique"),
    total_amount=("amount", "sum")
).reset_index()

application_status_pivot = final_df.groupby("last_application_status").agg(
    customer_count=("customer_id", "nunique")
).reset_index()

with pd.ExcelWriter("task3_final_dashboard_dataset.xlsx") as writer:
    final_df.to_excel(writer, sheet_name="final_dataset", index=False)
    segment_pivot.to_excel(writer, sheet_name="segment_pivot", index=False)
    df_vip.to_excel(writer, sheet_name="VIP", index=False)
    df_active.to_excel(writer, sheet_name="Active", index=False)
    df_risk.to_excel(writer, sheet_name="Risk", index=False)
    df_regular.to_excel(writer, sheet_name="Regular", index=False)

    df_kpi.to_excel(writer, sheet_name="kpi", index=False)
    city_pivot.to_excel(writer, sheet_name="city_pivot", index=False)
    loan_status_pivot.to_excel(writer, sheet_name="loan_status", index=False)
    channel_pivot.to_excel(writer, sheet_name="channels", index=False)
    activity_by_year.to_excel(writer, sheet_name="activity_by_year", index=False)
    application_status_pivot.to_excel(writer, sheet_name="applications_status", index=False)
